<a href="https://colab.research.google.com/github/paidasahithi26/SahithiPaida_INFO5731_Fall2024/blob/main/In_Class_Exercise_5%266_Feature_Extraction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [4]:
from google.colab import files
import pandas as pd

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))

print("Uploaded file name:", filename)

# ✅ create dataframe first
df = pd.read_csv(filename)

# now this works
print("\nColumns:", df.columns.tolist())


Saving product_reviews.txt to product_reviews (2).txt
Uploaded file name: product_reviews (2).txt


ParserError: Error tokenizing data. C error: Expected 1 fields in line 7, saw 2


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [8]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = next(iter(uploaded))

print("Uploaded file name:", filename)

# Try tab separator
df = pd.read_csv(filename, sep="\t")

print("\nColumns:", df.columns.tolist())
print("\nShape:", df.shape)
print("\nFirst 3 rows:")
print(df.head(3))

Saving product_reviews.txt to product_reviews (4).txt
Uploaded file name: product_reviews (4).txt

Columns: ['id', 'text']

Shape: (10, 2)

First 3 rows:
   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [9]:
# Write your Answer here (Q2)
import string

# word_count → number of words
df["word_count"] = df["text"].apply(lambda x: len(str(x).split()))

# char_count → number of characters
df["char_count"] = df["text"].apply(lambda x: len(str(x)))

# avg_word_len → average word length (ignore punctuation)
df["avg_word_len"] = df["text"].apply(
    lambda x: (
        sum(len(word.strip(string.punctuation)) for word in str(x).split())
        / max(len(str(x).split()), 1)
    )
)

# excl_count → number of '!'
df["excl_count"] = df["text"].apply(lambda x: str(x).count("!"))

# print required columns
print(df[["id", "word_count", "char_count", "avg_word_len", "excl_count"]])


   id  word_count  char_count  avg_word_len  excl_count
0   1           9          55      5.000000           1
1   2           7          51      5.571429           3
2   3           8          45      4.500000           0
3   4          10          56      4.500000           0
4   5           8          53      5.500000           1
5   6          10          51      4.000000           0
6   7           9          50      4.444444           0
7   8           6          40      5.500000           0
8   9           7          52      6.285714           0
9  10           7          48      5.571429           0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [10]:
# # Write your Answer here (Q3)
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Initialize CountVectorizer with English stop words
vectorizer = CountVectorizer(stop_words="english")

# Fit and transform the text column
X = vectorizer.fit_transform(df["text"])

# Vocabulary size
vocab_size = len(vectorizer.get_feature_names_out())
print("Vocabulary size:", vocab_size)

# Sum counts of each word across all documents
word_counts = X.toarray().sum(axis=0)

# Create a DataFrame for easy sorting
word_freq = pd.DataFrame({
    "word": vectorizer.get_feature_names_out(),
    "count": word_counts
})

# Top 10 words by total count
top_10 = word_freq.sort_values(by="count", ascending=False).head(10)
print("\nTop 10 words by total count:")
print(top_10)



Vocabulary size: 50

Top 10 words by total count:
       word  count
0   amazing      1
1       app      1
2   battery      1
3   blender      1
4       box      1
5       buy      1
6   charged      1
7     clear      1
8  crashing      1
9    creamy      1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [11]:
# Write your Answer here (Q4)
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Initialize CountVectorizer for bigrams
bigram_vectorizer = CountVectorizer(stop_words="english", ngram_range=(2,2))

# Fit and transform the text column
X_bigram = bigram_vectorizer.fit_transform(df["text"])

# Sum counts of each bigram across all documents
bigram_counts = X_bigram.toarray().sum(axis=0)

# Create DataFrame for easy sorting
bigram_freq = pd.DataFrame({
    "bigram": bigram_vectorizer.get_feature_names_out(),
    "count": bigram_counts
})

# Top 5 bigrams by total count
top_5_bigrams = bigram_freq.sort_values(by="count", ascending=False).head(5)
print("Top 5 bigrams by total count:")
print(top_5_bigrams)


Top 5 bigrams by total count:
              bigram  count
0     amazing screen      1
1          app keeps      1
2       battery life      1
3  blender smoothies      1
4        box damaged      1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [12]:
# Write your Answer here  (Q5)
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Initialize TF-IDF vectorizer (unigrams + bigrams)
tfidf_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))

# Fit and transform the text column
X_tfidf = tfidf_vectorizer.fit_transform(df["text"])

# Compute average TF-IDF for each term across all documents
avg_tfidf = X_tfidf.mean(axis=0).A1  # .A1 converts to 1D array

# Create a DataFrame for easy sorting
tfidf_df = pd.DataFrame({
    "term": tfidf_vectorizer.get_feature_names_out(),
    "avg_tfidf": avg_tfidf
})

# Top 5 terms by average TF-IDF
top_5_tfidf = tfidf_df.sort_values(by="avg_tfidf", ascending=False).head(5)
print("Top 5 terms by average TF-IDF:")
print(top_5_tfidf)


Top 5 terms by average TF-IDF:
                  term  avg_tfidf
25                easy   0.037796
13               clear   0.037796
61               setup   0.037796
38  instructions clear   0.037796
37        instructions   0.037796


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
